In [ ]:
RETRAIN = False


# Teacher-Student ViT-B/16 x224 Main

This notebook tests whether ViT-B/16 teacher-student distillation can match or approach PatchCore's performance.

**Key question:** Does ViT superiority come from the backbone, the scoring method, or both?


## Submission Context

- Dataset: `data/processed/x224/wm811k/metadata_50k_5pct.csv`
- Experiment config: `experiments/anomaly_detection/teacher_student/vit_b16/x224/main/train_config.toml`
- Artifact root: `experiments/anomaly_detection/teacher_student/vit_b16/x224/main/artifacts/ts_vit_b16_x224`
- Saved outputs:
  - checkpoints in `artifacts/ts_vit_b16_x224/checkpoints/`
  - CSV and JSON results in `artifacts/ts_vit_b16_x224/results/`
  - regenerated figures in `artifacts/ts_vit_b16_x224/plots/`


## Setup and Initialization


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.ts_distillation import build_ts_distillation_from_config
from wafer_defect.scoring import spatial_max, spatial_mean, topk_spatial_mean
REPO_ROOT


## Configuration and Artifact Setup


In [ ]:
ARTIFACT_DIR = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/vit_b16/x224/main/artifacts/ts_vit_b16_x224'
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/vit_b16/x224/main/train_config.toml'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
RESULTS_DIR = ARTIFACT_DIR / 'results'
EVALUATION_DIR = RESULTS_DIR / 'evaluation'
PLOTS_DIR = ARTIFACT_DIR / 'plots'
CHECKPOINT_PATH = CHECKPOINT_DIR / 'best_model.pt'
THRESHOLD_QUANTILE = 0.95
RUN_SCORE_SWEEP = False
SCORE_SWEEP_WEIGHTS = [(1.0, 1.0), (1.0, 0.0), (0.0, 1.0), (2.0, 1.0), (1.0, 2.0), (1.0, 0.5), (0.5, 1.0)]
SCORE_SWEEP_REDUCTIONS = [('mean', None), ('max', None), ('topk_mean', 0.01), ('topk_mean', 0.05), ('topk_mean', 0.1), ('topk_mean', 0.2)]
config = load_toml(CONFIG_PATH)
image_size = int(config['data'].get('image_size', 224))
batch_size = int(config['data'].get('batch_size', 256))
num_workers = int(config['data'].get('num_workers', 8))
requested_device = str(config['training'].get('device', 'auto'))
resolved_device_name = 'cuda' if requested_device == 'auto' and torch.cuda.is_available() else requested_device
device = torch.device(resolved_device_name)
for path in [CHECKPOINT_DIR, RESULTS_DIR, EVALUATION_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print(f'Artifact dir: {ARTIFACT_DIR}')
print(f'Config: {CONFIG_PATH}')
print(f'Device: {device}')
print(f'Checkpoint: {CHECKPOINT_PATH}')


## Training (if needed)


In [ ]:
def stream_command(command, cwd):
    print('Running:', ' '.join((str(part) for part in command)))
    process = subprocess.Popen([str(part) for part in command], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def move_if_exists(source, destination):
    source = Path(source)
    destination = Path(destination)
    if source.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            destination.unlink()
        source.replace(destination)

def normalize_run_artifacts(run_dir):
    run_dir = Path(run_dir)
    checkpoint_dir = run_dir / 'checkpoints'
    results_dir = run_dir / 'results'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)
    for filename in ['best_model.pt', 'latest_checkpoint.pt', 'last_model.pt']:
        root_file = run_dir / filename
        target_file = checkpoint_dir / filename
        if root_file.exists() and root_file.resolve() != target_file.resolve():
            move_if_exists(root_file, target_file)
    for checkpoint_file in run_dir.glob('checkpoint_epoch_*.pt'):
        move_if_exists(checkpoint_file, checkpoint_dir / checkpoint_file.name)
    for filename in ['history.json', 'summary.json']:
        root_file = run_dir / filename
        target_file = results_dir / filename
        if root_file.exists() and root_file.resolve() != target_file.resolve():
            move_if_exists(root_file, target_file)
normalize_run_artifacts(ARTIFACT_DIR)
if RETRAIN:
    stream_command([sys.executable, '-u', REPO_ROOT / 'scripts' / 'train_ts_distillation.py', '--config', CONFIG_PATH], cwd=REPO_ROOT)
    normalize_run_artifacts(ARTIFACT_DIR)
else:
    print(f'Skipping local training. Expecting checkpoint at {CHECKPOINT_PATH}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')


## Load Training History and Visualize


In [ ]:
history_path = RESULTS_DIR / 'history.json'
run_summary_path = RESULTS_DIR / 'summary.json'
if not history_path.exists():
    raise FileNotFoundError(f'History not found: {history_path}')
history_df = pd.read_json(history_path)
if run_summary_path.exists():
    run_summary = json.loads(run_summary_path.read_text(encoding='utf-8'))
else:
    checkpoint_summary = torch.load(CHECKPOINT_PATH, map_location='cpu')
    run_summary = {'best_epoch': int(checkpoint_summary.get('best_epoch', checkpoint_summary.get('epoch', 0))), 'best_val_loss': float(checkpoint_summary.get('best_val_loss', float('nan'))), 'student_map_scale': float(checkpoint_summary.get('student_map_scale', 1.0)), 'autoencoder_map_scale': float(checkpoint_summary.get('autoencoder_map_scale', 1.0)), 'history_rows': int(len(history_df))}
    run_summary_path.write_text(json.dumps(run_summary, indent=2), encoding='utf-8')
display(pd.Series(run_summary))
display(history_df.tail())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0], title='TS-ViT-B/16 Total Loss')
history_df.plot(x='epoch', y=['train_distillation_loss', 'train_autoencoder_loss', 'val_distillation_loss', 'val_autoencoder_loss'], ax=axes[1], title='Loss Components')
axes[0].set_ylabel('loss')
axes[1].set_ylabel('loss')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'training_curves.png', dpi=200, bbox_inches='tight')
plt.show()


## Load Model and Data


In [ ]:
metadata_path = REPO_ROOT / config['data']['metadata_csv']
metadata = pd.read_csv(metadata_path)
display(metadata.groupby(['split', 'is_anomaly']).size().rename('count').reset_index())
val_dataset = WaferMapDataset(metadata_path, split='val', image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split='test', image_size=image_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
print(f'val={len(val_dataset)}, test={len(test_dataset)}')
model = build_ts_distillation_from_config(config, image_size=image_size).to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
if 'student_map_scale' in checkpoint:
    model.student_map_scale.fill_(max(float(checkpoint['student_map_scale']), 1e-06))
if 'autoencoder_map_scale' in checkpoint:
    model.autoencoder_map_scale.fill_(max(float(checkpoint['autoencoder_map_scale']), 1e-06))
model.eval()
print('Loaded checkpoint successfully.')


## Evaluation Helpers


In [ ]:
def collect_scores_local(model, dataloader, device):
    rows = []
    model.eval()
    with torch.inference_mode():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            scores = model(inputs)
            for score, label in zip(scores.cpu().tolist(), labels.tolist()):
                rows.append({'score': float(score), 'is_anomaly': int(label)})
    return pd.DataFrame(rows)

def collect_normalized_maps(model, dataloader, device):
    student_maps = []
    auto_maps = []
    labels = []
    with torch.inference_mode():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            student_map, auto_map = model.raw_anomaly_maps(inputs)
            student_maps.append((student_map / model.student_map_scale.clamp_min(1e-06)).cpu())
            auto_maps.append((auto_map / model.autoencoder_map_scale.clamp_min(1e-06)).cpu())
            labels.append(batch_labels.cpu())
    return (torch.cat(student_maps, dim=0), torch.cat(auto_maps, dim=0), torch.cat(labels, dim=0).numpy())

def reduce_anomaly_map(anomaly_map, reduction, topk_ratio):
    if reduction == 'mean':
        return spatial_mean(anomaly_map)
    if reduction == 'max':
        return spatial_max(anomaly_map)
    return topk_spatial_mean(anomaly_map, topk_ratio=topk_ratio)

def plot_confusion_matrix(ax, labels, predictions, title):
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())
    tp = int(((labels == 1) & (predictions == 1)).sum())
    confusion = np.array([[tn, fp], [fn, tp]])
    ax.imshow(confusion, cmap='Blues')
    ax.set_xticks([0, 1], labels=['pred normal', 'pred anomaly'])
    ax.set_yticks([0, 1], labels=['true normal', 'true anomaly'])
    ax.set_title(title)
    for row_index in range(confusion.shape[0]):
        for column_index in range(confusion.shape[1]):
            ax.text(column_index, row_index, confusion[row_index, column_index], ha='center', va='center', color='#111111')


## Default Evaluation And Saved Figures


In [ ]:
summary_path = EVALUATION_DIR / 'summary.json'
val_scores_path = EVALUATION_DIR / 'val_scores.csv'
test_scores_path = EVALUATION_DIR / 'test_scores.csv'
threshold_sweep_path = EVALUATION_DIR / 'threshold_sweep.csv'
if RETRAIN:
    val_scores_df = collect_scores_local(model, val_loader, device)
    test_scores_df = collect_scores_local(model, test_loader, device)
    threshold = float(val_scores_df.loc[val_scores_df['is_anomaly'] == 0, 'score'].quantile(THRESHOLD_QUANTILE))
    labels = test_scores_df['is_anomaly'].to_numpy()
    scores = test_scores_df['score'].to_numpy()
    metrics = summarize_threshold_metrics(labels, scores, threshold)
    threshold_sweep_df, best_sweep = sweep_threshold_metrics(labels, scores)
    val_scores_df.to_csv(val_scores_path, index=False)
    test_scores_df.to_csv(test_scores_path, index=False)
    threshold_sweep_df.to_csv(threshold_sweep_path, index=False)
    evaluation_summary = {'model_type': 'ts_distillation', 'checkpoint': str(CHECKPOINT_PATH), 'threshold_quantile': THRESHOLD_QUANTILE, 'threshold': threshold, 'metrics_at_validation_threshold': metrics, 'best_threshold_sweep': best_sweep}
    summary_path.write_text(json.dumps(evaluation_summary, indent=2), encoding='utf-8')
else:
    if not summary_path.exists():
        raise FileNotFoundError(f'Evaluation summary not found: {summary_path}')
    if not val_scores_path.exists() or not test_scores_path.exists() or (not threshold_sweep_path.exists()):
        raise FileNotFoundError('Expected cached evaluation CSV files are missing. Enable RETRAIN to rebuild them.')
    evaluation_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    val_scores_df = pd.read_csv(val_scores_path)
    test_scores_df = pd.read_csv(test_scores_path)
    threshold_sweep_df = pd.read_csv(threshold_sweep_path)
display(pd.Series(evaluation_summary['metrics_at_validation_threshold']))
display(pd.Series(evaluation_summary['best_threshold_sweep']))
default_threshold = float(evaluation_summary['threshold'])
default_predictions = (test_scores_df['score'].to_numpy() > default_threshold).astype(int)
default_labels = test_scores_df['is_anomaly'].to_numpy()
fig, ax = plt.subplots(figsize=(8, 4.5))
normal_scores = test_scores_df.loc[test_scores_df['is_anomaly'] == 0, 'score']
anomaly_scores = test_scores_df.loc[test_scores_df['is_anomaly'] == 1, 'score']
ax.hist(normal_scores, bins=40, alpha=0.65, label='normal', color='#8d99ae', density=True)
ax.hist(anomaly_scores, bins=40, alpha=0.55, label='anomaly', color='#e76f51', density=True)
ax.axvline(default_threshold, color='#264653', linestyle='--', linewidth=2, label=f'threshold={default_threshold:.4f}')
ax.set_title('TS-ViT-B/16 Score Distribution')
ax.set_xlabel('anomaly score')
ax.set_ylabel('density')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
fig, ax = plt.subplots(figsize=(8, 4.5))
threshold_sweep_df.plot(x='threshold', y=['precision', 'recall', 'f1'], ax=ax)
ax.axvline(default_threshold, color='#264653', linestyle='--', linewidth=1.75, label='validation threshold')
ax.set_title('Threshold Sweep Metrics')
ax.set_ylabel('metric')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
plt.show()
fig, ax = plt.subplots(figsize=(5, 4.5))
plot_confusion_matrix(ax, default_labels, default_predictions, 'Confusion Matrix At Validation Threshold')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()


## Score Sweep And Comparison Plot


In [ ]:
score_sweep_path = EVALUATION_DIR / 'score_sweep_summary.csv'
selected_variant_path = EVALUATION_DIR / 'selected_score_variant.json'
if RUN_SCORE_SWEEP:
    val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(model, val_loader, device)
    test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(model, test_loader, device)
    score_sweep_rows = []
    for student_weight, auto_weight in SCORE_SWEEP_WEIGHTS:
        for reduction, topk_ratio in SCORE_SWEEP_REDUCTIONS:
            variant_name = f's{student_weight:g}_a{auto_weight:g}_{reduction}'
            if topk_ratio is not None:
                variant_name += f'_r{topk_ratio:.2f}'
            val_map = student_weight * val_student_maps + auto_weight * val_auto_maps
            test_map = student_weight * test_student_maps + auto_weight * test_auto_maps
            val_scores = reduce_anomaly_map(val_map, reduction, topk_ratio)
            test_scores = reduce_anomaly_map(test_map, reduction, topk_ratio)
            if hasattr(val_scores, 'numpy'):
                val_scores = val_scores.numpy()
            if hasattr(test_scores, 'numpy'):
                test_scores = test_scores.numpy()
            val_scores = np.asarray(val_scores).reshape(-1)
            test_scores = np.asarray(test_scores).reshape(-1)
            threshold = float(pd.Series(val_scores)[val_labels == 0].quantile(THRESHOLD_QUANTILE))
            metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
            _, best_sweep = sweep_threshold_metrics(test_labels, test_scores)
            score_sweep_rows.append({'name': variant_name, 'student_weight': student_weight, 'auto_weight': auto_weight, 'reduction': reduction, 'topk_ratio': np.nan if topk_ratio is None else topk_ratio, 'threshold': threshold, 'precision': metrics['precision'], 'recall': metrics['recall'], 'f1': metrics['f1'], 'auroc': metrics['auroc'], 'auprc': metrics['auprc'], 'best_sweep_f1': best_sweep['f1']})
    score_sweep_df = pd.DataFrame(score_sweep_rows).sort_values(['f1', 'auroc', 'auprc'], ascending=[False, False, False]).reset_index(drop=True)
    selected_variant = score_sweep_df.iloc[0].to_dict()
    score_sweep_df.to_csv(score_sweep_path, index=False)
    selected_variant_path.write_text(json.dumps(selected_variant, indent=2), encoding='utf-8')
else:
    if not score_sweep_path.exists() or not selected_variant_path.exists():
        raise FileNotFoundError('Expected cached score-sweep outputs are missing. Enable RUN_SCORE_SWEEP to rebuild them.')
    score_sweep_df = pd.read_csv(score_sweep_path)
    selected_variant = json.loads(selected_variant_path.read_text(encoding='utf-8'))
display(score_sweep_df.head(10).round(6))
display(pd.Series(selected_variant))
summary_comparison_df = pd.DataFrame([{'name': 'default_score', 'precision': evaluation_summary['metrics_at_validation_threshold']['precision'], 'recall': evaluation_summary['metrics_at_validation_threshold']['recall'], 'f1': evaluation_summary['metrics_at_validation_threshold']['f1'], 'auroc': evaluation_summary['metrics_at_validation_threshold']['auroc'], 'auprc': evaluation_summary['metrics_at_validation_threshold']['auprc'], 'best_sweep_f1': evaluation_summary['best_threshold_sweep']['f1']}, {'name': selected_variant['name'], 'precision': selected_variant['precision'], 'recall': selected_variant['recall'], 'f1': selected_variant['f1'], 'auroc': selected_variant['auroc'], 'auprc': selected_variant['auprc'], 'best_sweep_f1': selected_variant['best_sweep_f1']}])
summary_comparison_df.to_csv(EVALUATION_DIR / 'score_variant_comparison.csv', index=False)
display(summary_comparison_df.round(6))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(summary_comparison_df['name'], summary_comparison_df['f1'], color=['#8d99ae', '#2a9d8f'])
axes[0].set_title('Validation-Threshold F1')
axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(summary_comparison_df['name'], summary_comparison_df['auroc'], color=['#8d99ae', '#2a9d8f'])
axes[1].set_title('AUROC')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'score_variant_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## Defect-Type Breakdown


In [ ]:
defect_breakdown_path = EVALUATION_DIR / f"{selected_variant['name']}_defect_breakdown.csv"
if defect_breakdown_path.exists():
    defect_breakdown_df = pd.read_csv(defect_breakdown_path)
else:
    val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(model, val_loader, device)
    test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(model, test_loader, device)
    student_weight = float(selected_variant['student_weight'])
    auto_weight = float(selected_variant['auto_weight'])
    reduction = str(selected_variant['reduction'])
    topk_ratio_value = selected_variant.get('topk_ratio', np.nan)
    topk_ratio = None if pd.isna(topk_ratio_value) else float(topk_ratio_value)
    selected_threshold = float(selected_variant['threshold'])
    selected_test_map = student_weight * test_student_maps + auto_weight * test_auto_maps
    selected_test_scores = reduce_anomaly_map(selected_test_map, reduction, topk_ratio)
    if hasattr(selected_test_scores, 'numpy'):
        selected_test_scores = selected_test_scores.numpy()
    selected_test_scores = np.asarray(selected_test_scores).reshape(-1)
    analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
    analysis_df['score'] = selected_test_scores
    analysis_df['predicted_anomaly'] = (analysis_df['score'] > selected_threshold).astype(int)
    defect_breakdown_df = analysis_df.loc[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean'), median_score=('score', 'median')).reset_index()
    defect_breakdown_df['detected'] = defect_breakdown_df['detected'].astype(int)
    defect_breakdown_df['missed'] = defect_breakdown_df['count'] - defect_breakdown_df['detected']
    defect_breakdown_df['recall'] = defect_breakdown_df['detected'] / defect_breakdown_df['count']
    defect_breakdown_df = defect_breakdown_df.sort_values(['recall', 'count', 'defect_type'], ascending=[True, False, True]).reset_index(drop=True)
    defect_breakdown_df.to_csv(defect_breakdown_path, index=False)
display(defect_breakdown_df)
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(defect_breakdown_df['defect_type'], defect_breakdown_df['recall'], color='#457b9d')
ax.set_ylim(0.0, 1.05)
ax.set_title('Recall By Defect Type For Selected Score Variant')
ax.set_ylabel('recall')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
plt.show()
